# Gemma 4 31B Instruct — Kaggle TPU v5e-8 production demo

Runs the public Gemma 4 31B inference server on a Kaggle TPU v5e-8
accelerator, then calls its public REST interface. Only files that exist in
the public repository are used.

## Before you run

1. Enable the **TPU v5e-8 / v5litepod-8** accelerator in the session options.
2. Attach the **Gemma 4 31B** Keras model preset as an input dataset.
3. Turn on the Internet option; this notebook clones the repository and
   installs the runtime with pip.
4. Expect a slow first request. The qualified run needed about 541 seconds to
   load the model and compile the first generation graph. Keep the server
   process alive between requests so later compatible requests reuse the
   compiled executable.

Run **Run All** from the top; the cells are ordered. If a cell fails, see
[docs/troubleshooting.md](../docs/troubleshooting.md).

In [ ]:
import json, os, secrets, subprocess, sys
from pathlib import Path

REPO_URL = 'https://github.com/dangkhoa2016/KerasHub-Gemma4-31B-IT-Kaggle-TPU-v5e8-Text-Vision.git'
REPO_DIR = Path('/kaggle/working/gemma4-31b-tpu')
MODEL_PRESET = 'gemma4_instruct_31b'
PUBLIC_FILES = (
    'scripts/configure_kaggle_tpu.sh',
    'scripts/check_tpu_preflight.py',
    'scripts/resolve_kaggle_model_path.py',
    'scripts/ensure_libtpu.py',
    'scripts/start.sh',
    'scripts/status.sh',
    'scripts/stop.sh',
    'scripts/wait_ready.py',
    'clients/python/gemma4_client.py',
    '.env.example',
)


def sh(args, **kwargs):
    print('$', ' '.join(str(a) for a in args))
    return subprocess.run([str(a) for a in args], **kwargs)


print('MODEL_PRESET=' + MODEL_PRESET)
print('SESSION=' + os.environ.get('KAGGLE_KERNEL_RUN_TYPE', 'not-kaggle'))

## 1. Check out the public source

Clones on first run, then fetches, so re-running this cell reuses the
existing checkout.

In [ ]:
if not (REPO_DIR / '.git').exists():
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    sh(['git', 'clone', REPO_URL, REPO_DIR], check=True)
sh(['git', '-C', REPO_DIR, 'fetch', '--prune', 'origin'], check=True)
sh(['git', '-C', REPO_DIR, 'checkout', 'origin/main'], check=True)
sh(['git', '-C', REPO_DIR, 'reset', '--hard', 'origin/main'], check=True)

SERVER_SHA = subprocess.check_output(
    ['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'], text=True
).strip()
missing = [n for n in PUBLIC_FILES if not (REPO_DIR / n).is_file()]
assert not missing, f'Public repository files are missing: {missing}'
print('SERVER_SHA=' + SERVER_SHA)
print('PUBLIC_FILES_PRESENT=' + str(len(PUBLIC_FILES)))

## 2. Install the public runtime

Kaggle already ships a working JAX/JAXLIB accelerator stack, so pip installs
only `requirements.txt`. `scripts/ensure_libtpu.py` keeps an installed
`libtpu` and otherwise installs the version pinned in `requirements-tpu.txt`.

In [ ]:
sh([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
    '--root-user-action=ignore', '--no-warn-conflicts', '--progress-bar', 'off',
    '-r', REPO_DIR / 'requirements.txt'], cwd=REPO_DIR, check=True)
env = os.environ.copy()
env['PYTHONPATH'] = str(REPO_DIR / 'src')
sh([sys.executable, 'scripts/ensure_libtpu.py'], cwd=REPO_DIR, env=env, check=True)

## 3. Resolve the attached model

A missing dataset fails here with a clear message instead of a late
model-load error.


In [ ]:
resolved = sh([sys.executable, 'scripts/resolve_kaggle_model_path.py',
               '--input-root', '/kaggle/input', '--preset-name', MODEL_PRESET],
              cwd=REPO_DIR, env=env, text=True, capture_output=True)
if resolved.returncode != 0:
    print(resolved.stdout, end='')
    print(resolved.stderr, end='', file=sys.stderr)
    raise SystemExit('Attach the Gemma 4 31B model dataset and run this cell again.')

MODEL_PATH = resolved.stdout.strip().splitlines()[-1].strip()
assert Path(MODEL_PATH).exists(), f'Model path does not exist: {MODEL_PATH}'
print('MODEL_PATH=' + MODEL_PATH)

## 4. Configure secrets and the TPU runtime

A random API key and restart secret are generated for this session and
written to `.env`, which is excluded from version control. Nothing secret is
printed. `scripts/configure_kaggle_tpu.sh` applies the public Kaggle TPU
discovery fallback when the platform metadata service is unavailable.

In [ ]:
keys = {'API_KEY': secrets.token_urlsafe(24), 'RESTART_SECRET': secrets.token_urlsafe(24)}
lines = []
for line in (REPO_DIR / '.env.example').read_text(encoding='utf-8').splitlines():
    name = line.split('=', 1)[0]
    if name in keys:
        lines.append(f'{name}={keys[name]}')
    elif name == 'MODEL_PATH':
        lines.append('MODEL_PATH=' + MODEL_PATH)
    else:
        lines.append(line)
(REPO_DIR / '.env').write_text('\n'.join(lines) + '\n', encoding='utf-8')
print('ENV_WRITTEN=.env (API_KEY and RESTART_SECRET generated, not printed)')
sh(['bash', 'scripts/configure_kaggle_tpu.sh'], cwd=REPO_DIR, env=env, check=True)

## 5. TPU preflight

Confirms eight TPU devices are visible through JAX before the server starts,
so a missing accelerator fails fast.

In [ ]:
preflight = sh([sys.executable, 'scripts/check_tpu_preflight.py'], cwd=REPO_DIR, env=env, text=True)
print(preflight.stdout, end='')
if preflight.returncode != 0:
    raise SystemExit('TPU preflight failed. Enable the TPU v5e-8 accelerator and restart the session.')
print('TPU_PREFLIGHT=PASS')

## 6. Start the server and wait for readiness

`scripts/start.sh` launches one long-lived coordinator that owns the TPU
worker process. `scripts/wait_ready.py` blocks until `/health/ready`. The model
load and first compile are the slow part of this cell.

In [ ]:
sh(['bash', 'scripts/start.sh'], cwd=REPO_DIR, env=env, check=True)
ready_env = dict(env, READY_URL='http://127.0.0.1:7860/health/ready', READY_TIMEOUT='3600')
waited = sh([sys.executable, 'scripts/wait_ready.py'], cwd=REPO_DIR, env=ready_env, text=True)
if waited.returncode != 0:
    print((REPO_DIR / 'logs' / 'server.stdout.log').read_text()[-4000:])
    raise SystemExit('Server did not become ready.')
print('SERVER_READY=PASS')

## 7. Text request through the public REST interface

Calls the shipped Python client, which uses `POST /generate/async` and polls
`GET /result/<job_id>`. The API key is read from `.env` and never printed.

In [ ]:
settings = dict(
    line.split('=', 1) for line in
    (REPO_DIR / '.env').read_text(encoding='utf-8').splitlines()
    if '=' in line
)


def request(prompt, image=None):
    args = [sys.executable, 'clients/python/gemma4_client.py',
            '--url', 'http://127.0.0.1:7860', '--api-key', settings['API_KEY'],
            '--prompt', prompt, '--max-new-tokens', '96']
    if image:
        args += ['--image', image]
    done = sh(args, cwd=REPO_DIR, env=env, text=True, capture_output=True)
    print(done.stdout, end='')
    if done.returncode != 0:
        print(done.stderr, end='', file=sys.stderr)
        raise SystemExit('Generation request failed.')


request('In three sentences, explain model parallelism on eight TPU devices.')
print('TEXT_REQUEST=PASS')

## 8. Optional: image-conditioned generation

Skipped when no test image is found under `/kaggle/input`.

In [ ]:
suffixes = {'.jpg', '.jpeg', '.png', '.webp'}
root = Path('/kaggle/input')
sample = next((p for p in sorted(root.rglob('*'))
              if p.suffix.lower() in suffixes and p.is_file()), None) if root.is_dir() else None

if sample is None:
    print('VISION_REQUEST=SKIPPED (no image found under /kaggle/input)')
else:
    print('IMAGE_PATH=' + str(sample))
    request('Describe this image in one sentence.', image=sample)
    print('VISION_REQUEST=PASS')

## 9. Session status

The server stays alive for the life of the Kaggle kernel so the compiled
executable is reused by later compatible requests. Stop it explicitly
with `bash scripts/stop.sh` when you no longer need it.


In [ ]:
sh(['bash', 'scripts/status.sh'], cwd=REPO_DIR, env=env, check=True)
print(json.dumps({
    'server_sha': SERVER_SHA,
    'model_path': MODEL_PATH,
    'tpu_devices_expected': 8,
    'tpu_preflight': 'PASS',
    'server_ready': 'PASS',
    'text_request': 'PASS',
    'vision_request': 'SKIPPED' if sample is None else 'PASS',
}, indent=2))